# 1. Đọc dữ liệu đã chia sẵn; Signed-log xử lý trong bộ nhớ


In [ ]:
from pathlib import Path
import inspect
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    precision_recall_fscore_support,
    confusion_matrix,
)

EXPERIMENT_ID = "E13"
COMPARE_ID = "E12"
TARGET = "Attack_type"
RANDOM_STATE = 42
N_SPLITS = 3

CURRENT = Path.cwd().resolve()
ROOT = next(
    (
        parent
        for parent in [CURRENT, *CURRENT.parents]
        if (parent / "project").is_dir()
        and (parent / "splitted data").is_dir()
    ),
    None,
)
if ROOT is None:
    raise FileNotFoundError(
        "Không tìm thấy thư mục sample_project có project và splitted data."
    )

DATA_DIR = ROOT / "splitted data"
TRAIN_PATH = DATA_DIR / "train.csv"
VALID_PATH = DATA_DIR / "validation.csv"
for data_path in (TRAIN_PATH, VALID_PATH):
    if not data_path.is_file():
        raise FileNotFoundError(data_path)

train_df = pd.read_csv(TRAIN_PATH)
valid_df = pd.read_csv(VALID_PATH)
print("=== THỰC NGHIỆM E13 ===")
print("MLP + Signed-log + StandardScaler; hidden layers (128, 64)")
print("Train:", train_df.shape)
print("Validation:", valid_df.shape)

NUMERIC_COLUMNS = [
    column
    for column in train_df.select_dtypes(include=np.number).columns
    if column not in {TARGET, "Unnamed: 0", "group_id"}
]
train_df.loc[:, NUMERIC_COLUMNS] = np.sign(
    train_df[NUMERIC_COLUMNS]
) * np.log1p(np.abs(train_df[NUMERIC_COLUMNS]))
valid_df.loc[:, NUMERIC_COLUMNS] = np.sign(
    valid_df[NUMERIC_COLUMNS]
) * np.log1p(np.abs(valid_df[NUMERIC_COLUMNS]))
print("Đã áp dụng Signed-log trong bộ nhớ cho", len(NUMERIC_COLUMNS), "đặc trưng số.")

display(train_df.head(3))


## Tiền xử lý riêng cho E13

Chạy notebook `E13_MLP_SignedLog2_Processing.ipynb` trước. Notebook áp dụng Signed-log cho đặc trưng số của Train và Validation trong `splitted data`.

StandardScaler và OneHotEncoder được giữ trong Pipeline thực nghiệm để fit riêng trên phần Train của mỗi fold CV. E13 dùng kiến trúc MLP (128, 64).


# 2. Xây dựng Pipeline Signed-Log + StandardScaler + MLP (128,64)

In [ ]:
# ========================================
# 1. TÁCH ĐẶC TRƯNG VÀ NHÃN
# ========================================

DROP_COLS = [
    TARGET,
    "Unnamed: 0",
    "group_id"
]

CATEGORICAL_FEATURES = [
    "proto",
    "service"
]

if (
    TARGET not in train_df.columns
    or TARGET not in valid_df.columns
):
    raise ValueError(
        f"Thiếu cột nhãn {TARGET}"
    )

if (
    train_df[TARGET].isna().any()
    or valid_df[TARGET].isna().any()
):
    raise ValueError("Nhãn chứa giá trị thiếu")

X = train_df.drop(
    columns=DROP_COLS,
    errors="ignore"
).copy()

y = train_df[TARGET].astype(str).copy()

X_valid = valid_df.drop(
    columns=DROP_COLS,
    errors="ignore"
).copy()

y_valid = valid_df[TARGET].astype(str).copy()

# Đồng bộ thứ tự đặc trưng
if set(X.columns) != set(X_valid.columns):
    raise ValueError(
        "Train và Validation không có cùng đặc trưng"
    )

X_valid = X_valid[X.columns]

# ========================================
# 2. XÁC ĐỊNH CÁC LOẠI ĐẶC TRƯNG
# ========================================

missing_categorical = [
    col for col in CATEGORICAL_FEATURES
    if col not in X.columns
]

if missing_categorical:
    raise ValueError(
        f"Thiếu cột categorical: {missing_categorical}"
    )

NUMERIC_FEATURES = X.select_dtypes(
    include=np.number
).columns.tolist()

unknown_features = sorted(
    set(X.columns)
    - set(NUMERIC_FEATURES)
    - set(CATEGORICAL_FEATURES)
)

if unknown_features:
    raise ValueError(
        f"Cột chưa được xử lý: {unknown_features}"
    )

# ========================================
# 3. KIỂM TRA DỮ LIỆU
# ========================================

if X.isna().any().any() or X_valid.isna().any().any():
    raise ValueError(
        "Dữ liệu còn NaN, cần kiểm tra các tập đã chia sẵn"
    )

if not np.isfinite(
    X[NUMERIC_FEATURES].to_numpy(dtype=float)
).all():
    raise ValueError("Train có giá trị vô hạn")

if not np.isfinite(
    X_valid[NUMERIC_FEATURES].to_numpy(dtype=float)
).all():
    raise ValueError("Validation có giá trị vô hạn")

LABELS = sorted(y.unique().tolist())

if not set(y_valid.unique()).issubset(set(LABELS)):
    raise ValueError(
        "Validation chứa nhãn không có trong Train"
    )

if y.value_counts().min() < N_SPLITS:
    raise ValueError(
        "Có lớp không đủ 3 mẫu cho 3-fold CV"
    )

# ========================================
# 4. HÀM SIGNED-LOG
# ========================================

def signed_log(values):
    """
    Signed-log:
    sign(x) * log(1 + abs(x))
    """

    values = np.asarray(
        values,
        dtype=np.float64
    )

    return (
        np.sign(values)
        * np.log1p(np.abs(values))
    )


# ========================================
# 5. HÀM TẠO PIPELINE E13
# ========================================

def make_e13_pipeline():

    ohe_args = {
        "handle_unknown": "ignore"
    }

    # Tương thích các phiên bản sklearn
    if "sparse_output" in inspect.signature(
        OneHotEncoder
    ).parameters:
        ohe_args["sparse_output"] = False
    else:
        ohe_args["sparse"] = False

    # Signed-log đã được thực hiện trong bộ nhớ ở ô đọc dữ liệu.
    # StandardScaler được fit riêng theo từng fold CV.
    numeric_pipeline = Pipeline([
        ("scaler", StandardScaler())
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                NUMERIC_FEATURES
            ),
            (
                "categorical",
                OneHotEncoder(**ohe_args),
                CATEGORICAL_FEATURES
            )
        ],
        remainder="drop",
        sparse_threshold=0
    )

    # MLP E13: 2 lớp ẩn (128, 64)
    mlp = MLPClassifier(
        hidden_layer_sizes=(128, 64),
        activation="relu",
        solver="adam",
        alpha=0.0001,
        learning_rate_init=0.001,
        batch_size=512,
        max_iter=60,
        n_iter_no_change=8,
        early_stopping=False,
        random_state=RANDOM_STATE,
        verbose=False
    )

    return Pipeline([
        ("preprocess", preprocessor),
        ("model", mlp)
    ])


# ========================================
# 6. HÀM ĐÁNH GIÁ
# ========================================

METRICS = [
    "accuracy",
    "macro_f1",
    "weighted_f1",
    "macro_precision",
    "macro_recall"
]


def evaluate(y_true, y_pred):

    return {
        "accuracy": accuracy_score(
            y_true, y_pred
        ),

        "macro_f1": f1_score(
            y_true, y_pred,
            labels=LABELS,
            average="macro",
            zero_division=0
        ),

        "weighted_f1": f1_score(
            y_true, y_pred,
            labels=LABELS,
            average="weighted",
            zero_division=0
        ),

        "macro_precision": precision_score(
            y_true, y_pred,
            labels=LABELS,
            average="macro",
            zero_division=0
        ),

        "macro_recall": recall_score(
            y_true, y_pred,
            labels=LABELS,
            average="macro",
            zero_division=0
        )
    }


# ========================================
# 7. KIỂM TRA CẤU HÌNH
# ========================================

print("=== CẤU HÌNH E13 ===")

print("Train:", len(X))
print("Validation:", len(X_valid))
print("Số lớp:", len(LABELS))
print("Số đặc trưng số:", len(NUMERIC_FEATURES))
print("Categorical:", CATEGORICAL_FEATURES)

print("\n=== PIPELINE E13 ===")
print(make_e13_pipeline())

print("\n=== KIỂM TRA SIGNED-LOG ===")

sample_values = np.array(
    [-100, -10, -1, 0, 1, 10, 100],
    dtype=float
)

display(pd.DataFrame({
    "original": sample_values,
    "signed_log": signed_log(sample_values)
}))

### Tiền xử lý MLP E13

- Notebook Processing áp dụng Signed-log: `sign(x) × ln(1 + |x|)` cho đặc trưng số.
- Pipeline thực nghiệm tiếp tục dùng StandardScaler và OneHotEncoder.
- Các bộ biến đổi được fit bên trong từng fold Cross-Validation.


# 3. Huấn luyện E13 bằng 3-ford Cross-Validation

In [ ]:
# ========================================
# 3-FOLD CROSS-VALIDATION E13
# ========================================

cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

fold_rows = []
per_class_rows = []
loss_curves = {}

for fold, (fit_idx, cv_idx) in enumerate(
    cv.split(X, y),
    start=1
):

    print("\n" + "=" * 55)
    print(f"E13 - FOLD {fold}/{N_SPLITS}")
    print("=" * 55)

    # Dữ liệu Train trong fold
    X_fit = X.iloc[fit_idx]
    y_fit = y.iloc[fit_idx]

    # Dữ liệu kiểm định trong fold
    X_cv = X.iloc[cv_idx]
    y_cv = y.iloc[cv_idx]

    # Mỗi fold tạo Pipeline mới
    fold_pipeline = make_e13_pipeline()

    start_time = time.perf_counter()

    # Huấn luyện
    fold_pipeline.fit(X_fit, y_fit)

    elapsed = time.perf_counter() - start_time

    # Dự đoán
    pred_cv = fold_pipeline.predict(X_cv)
    pred_train = fold_pipeline.predict(X_fit)

    # Tính 5 chỉ số
    cv_scores = evaluate(y_cv, pred_cv)
    train_scores = evaluate(y_fit, pred_train)

    trained_mlp = fold_pipeline.named_steps["model"]

    # Lưu Training Loss
    loss_curves[fold] = list(
        trained_mlp.loss_curve_
    )

    # Lưu kết quả fold
    fold_rows.append({
        "fold": fold,
        "n_train": len(X_fit),
        "n_cv": len(X_cv),
        "epochs": int(trained_mlp.n_iter_),
        "training_time_sec": float(elapsed),
        "final_train_loss": float(trained_mlp.loss_),

        **cv_scores,

        **{
            f"train_{m}": train_scores[m]
            for m in METRICS
        }
    })

    # Tính chỉ số theo từng lớp
    p, r, f, support = (
        precision_recall_fscore_support(
            y_cv,
            pred_cv,
            labels=LABELS,
            zero_division=0
        )
    )

    for label, precision, recall, f1_value, n in zip(
        LABELS, p, r, f, support
    ):

        per_class_rows.append({
            "fold": fold,
            "class": label,
            "precision": float(precision),
            "recall": float(recall),
            "f1": float(f1_value),
            "support": int(n)
        })

    print("Số mẫu Train:", len(X_fit))
    print("Số mẫu CV:", len(X_cv))
    print("Số epoch:", trained_mlp.n_iter_)
    print(f"Thời gian: {elapsed:.2f} giây")

    print("\n=== 5 CHỈ SỐ FOLD ===")

    for metric in METRICS:
        print(
            f"{metric:18s}: {cv_scores[metric]:.6f}"
        )

# ========================================
# TỔNG HỢP KẾT QUẢ
# ========================================

cv_results = pd.DataFrame(fold_rows)
cv_per_class = pd.DataFrame(per_class_rows)

cv_summary = pd.DataFrame({
    "metric": METRICS,

    "mean": [
        float(cv_results[m].mean())
        for m in METRICS
    ],

    "std": [
        float(cv_results[m].std(ddof=1))
        for m in METRICS
    ]
})

cv_per_class_summary = (
    cv_per_class
    .groupby("class", as_index=False)
    .agg(
        precision_mean=("precision", "mean"),
        recall_mean=("recall", "mean"),
        f1_mean=("f1", "mean"),
        support_total=("support", "sum")
    )
    .sort_values("f1_mean")
)

print("\n=== KẾT QUẢ 3 FOLD ===")

display(
    cv_results[
        [
            "fold",
            *METRICS,
            "epochs",
            "training_time_sec"
        ]
    ].round(6)
)

print("\n=== TRUNG BÌNH 3 FOLD ===")
display(cv_summary.round(6))

print("\n=== 5 LỚP CÓ F1 THẤP NHẤT ===")

display(
    cv_per_class_summary
    .head(5)
    .round(6)
)

# ========================================
# LƯU KẾT QUẢ VÀO FOLDER E13
# ========================================





print("\nĐã lưu kết quả 3-fold CV.")

### Huấn luyện E13 bằng 3-fold Cross-Validation

**Mục tiêu:**

Đánh giá mô hình MLP (128, 64) sử dụng
Signed-log và StandardScaler bằng phương pháp
3-fold Cross-Validation.

**Quy trình:**
- Sử dụng StratifiedKFold với 3 fold.
- Mỗi fold khởi tạo một Pipeline mới.
- Huấn luyện trên 2 fold.


# 4. Huấn luyện toàn bộ Train và đánh giá Validation

In [ ]:
# ========================================
# 1. HUẤN LUYỆN TOÀN BỘ TRAIN
# ========================================

final_model = make_e13_pipeline()

start_time = time.perf_counter()

final_model.fit(X, y)

full_train_time = (
    time.perf_counter() - start_time
)

# Dự đoán
pred_train = final_model.predict(X)
pred_valid = final_model.predict(X_valid)

# Tính chỉ số
train_scores = evaluate(y, pred_train)
valid_scores = evaluate(y_valid, pred_valid)

final_mlp = final_model.named_steps["model"]

print("=== E13 - FULL TRAIN ===")
print("Số mẫu Train:", len(X))
print("Số epoch:", final_mlp.n_iter_)
print(f"Thời gian: {full_train_time:.2f} giây")
print(f"Training Loss: {final_mlp.loss_:.6f}")

# ========================================
# 2. BẢNG TRAIN / VALIDATION
# ========================================

validation_metrics = pd.DataFrame({
    "metric": METRICS,

    "train": [
        train_scores[m]
        for m in METRICS
    ],

    "validation": [
        valid_scores[m]
        for m in METRICS
    ]
})

validation_metrics["train_valid_gap"] = (
    validation_metrics["train"]
    - validation_metrics["validation"]
)

print("\n=== TRAIN / VALIDATION ===")

display(validation_metrics.round(6))

# ========================================
# 3. ĐÁNH GIÁ THEO TỪNG LỚP
# ========================================

p, r, f, support = (
    precision_recall_fscore_support(
        y_valid,
        pred_valid,
        labels=LABELS,
        zero_division=0
    )
)

validation_per_class = pd.DataFrame({
    "class": LABELS,
    "precision": p,
    "recall": r,
    "f1": f,
    "support": support
})

print("\n=== KẾT QUẢ THEO TỪNG LỚP ===")

display(
    validation_per_class
    .sort_values("f1")
    .round(6)
)

# ========================================
# 4. CONFUSION MATRIX
# ========================================

cm = confusion_matrix(
    y_valid,
    pred_valid,
    labels=LABELS
)

cm_df = pd.DataFrame(
    cm,
    index=LABELS,
    columns=LABELS
)

# ========================================
# 5. TỔNG HỢP CV VÀ VALIDATION CỦA E13
# ========================================

cv_vs_validation = (
    cv_summary[
        ["metric", "mean", "std"]
    ]
    .merge(
        validation_metrics[
            ["metric", "validation"]
        ],
        on="metric"
    )
)

cv_vs_validation["validation_minus_cv_mean"] = (
    cv_vs_validation["validation"]
    - cv_vs_validation["mean"]
)

print("\n=== CV / VALIDATION E13 ===")

display(cv_vs_validation.round(6))

# ========================================
# 6. LƯU KẾT QUẢ
# ========================================





print("\nĐã lưu kết quả Validation.")

### Huấn luyện toàn bộ Train và đánh giá Validation

**Mục tiêu:**

Đánh giá khả năng phân loại của mạng MLP
hai lớp ẩn (128, 64) sau khi dữ liệu được
xử lý bằng Signed-log và StandardScaler.

**Quy trình:**
- Khởi tạo một Pipeline E13 mới.
- Huấn luyện trên toàn bộ tập Train.
- Dự đoán trên tập Validation.
- Tính đầy đủ 5 chỉ số đánh giá.
- Đánh giá Precision, Recall và F1 theo từng lớp.

# 5. Vẽ biểu đồ kết quả 

In [ ]:
# ========================================
# 5.1. TRAINING LOSS CỦA 3 FOLD
# ========================================

fig, ax = plt.subplots(figsize=(9, 5))

for fold, losses in loss_curves.items():

    ax.plot(
        range(1, len(losses) + 1),
        losses,
        label=f"Fold {fold}"
    )

ax.set_xlabel("Epoch")
ax.set_ylabel("Training Loss")
ax.set_title("E13 - Training Loss 3-fold")

ax.legend()
fig.tight_layout()


plt.show()


# ========================================
# 5.2. TRAINING LOSS FULL TRAIN
# ========================================

fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(
    range(1, len(final_mlp.loss_curve_) + 1),
    final_mlp.loss_curve_
)

ax.set_xlabel("Epoch")
ax.set_ylabel("Training Loss")
ax.set_title("E13 - Full Train Loss")

fig.tight_layout()


plt.show()


# ========================================
# 5.3. CONFUSION MATRIX VALIDATION
# ========================================

row_totals = cm.sum(
    axis=1,
    keepdims=True
)

cm_normalized = np.divide(
    cm,
    row_totals,
    out=np.zeros_like(cm, dtype=float),
    where=row_totals != 0
)

fig, ax = plt.subplots(figsize=(12, 10))

im = ax.imshow(
    cm_normalized,
    cmap="Blues",
    vmin=0,
    vmax=1
)

ax.set_xticks(range(len(LABELS)))
ax.set_yticks(range(len(LABELS)))

ax.set_xticklabels(
    LABELS,
    rotation=75,
    ha="right"
)

ax.set_yticklabels(LABELS)

ax.set_xlabel("Predicted Label")
ax.set_ylabel("True Label")

ax.set_title(
    "E13 - Validation Confusion Matrix"
)

fig.colorbar(
    im,
    ax=ax,
    label="Tỷ lệ theo lớp thực tế"
)

fig.tight_layout()


plt.show()


# ========================================
# 5.4. F1-SCORE THEO TỪNG LỚP
# ========================================

f1_plot = (
    validation_per_class
    .sort_values("f1")
)

fig, ax = plt.subplots(figsize=(10, 7))

ax.barh(
    f1_plot["class"],
    f1_plot["f1"]
)

ax.set_xlim(0, 1)

ax.set_xlabel("F1-score")
ax.set_ylabel("Attack_type")

ax.set_title(
    "E13 - Validation F1-score theo từng lớp"
)

fig.tight_layout()


plt.show()




### 5. Trực quan hóa kết quả E13

**Các biểu đồ:**
- Training Loss của 3 fold.
- Training Loss trên toàn bộ Train.
- Confusion Matrix trên Validation.
- F1-score theo từng lớp trên Validation.

**Ý nghĩa:**

Training Loss giúp quan sát quá trình
tối ưu của mạng MLP (128, 64).

Confusion Matrix giúp xác định các trường
hợp phân loại đúng và nhầm lẫn giữa các nhãn.

Biểu đồ F1-score giúp nhận diện những
lớp có khả năng dự đoán còn hạn chế.

## 6. Tổng hợp kết quả trong notebook

Các chỉ số được hiển thị trực tiếp; notebook không xuất file vào `result`.


In [ ]:
print("Kết quả được giữ trong notebook; không ghi file vào result.")
display(cv_summary)
display(validation_metrics)
display(cv_vs_validation)
